# Cross-encoder member on a Kaggle T4 (`src/cross_encoder.py`)

Needs **no raw data**: only the text pairs exported on the laptop.

**Laptop, once:**
```
python src/cross_encoder.py export --split ce_train
python src/cross_encoder.py export --split local_val
python src/cross_encoder.py export --split test
```
Upload `cache/ce/{ce_train,local_val,test}.parquet` as a **private** Kaggle Dataset named `mlc26-ce`.

**Kaggle settings:** GPU T4, Internet on, secret `GH_TOKEN`.

**Output:** `cache/{local_val,test}/ce_scores.parquet` under `/kaggle/working`. Download both into the laptop's `cache/<split>/`, then rerun `match.py --cv` / `--fit` / `--predict`; stage 2 picks up the `ce` columns automatically.

In [ ]:
BRANCH = 'main'
INPUT = '/kaggle/input/mlc26-ce'
MAX_PAIRS = 600_000   # training pairs; the log prints pairs/s, so adjust to the time you have
import os, subprocess
from kaggle_secrets import UserSecretsClient
REPO = '/kaggle/working/MLC26'
if not os.path.exists(REPO):
    token = UserSecretsClient().get_secret('GH_TOKEN')
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@github.com/rishang19dx/AMZN_MLC2026.git', REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', 'https://github.com/rishang19dx/AMZN_MLC2026.git'], check=True)
PROJ = f'{REPO}/code/business_entity_resolution'
os.environ['BER_CACHE_DIR'] = '/kaggle/working/cache'
!git -C {REPO} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Kaggle ships torch + transformers; add what the DeBERTa-v3 tokenizer and our I/O need
!pip install -q sentencepiece protobuf duckdb
!mkdir -p /kaggle/working/cache/ce && cp {INPUT}/*.parquet /kaggle/working/cache/ce/ && ls -la /kaggle/working/cache/ce

In [ ]:
!cd {PROJ} && python src/cross_encoder.py train --max-pairs {MAX_PAIRS}

In [ ]:
!cd {PROJ} && python src/cross_encoder.py score --split local_val && python src/cross_encoder.py score --split test
!ls -la /kaggle/working/cache/local_val /kaggle/working/cache/test